# Chapter 4.4: a star schema with state as a first class field

One fact table of dispatches, four dimensions: state, rate version, SKU and date.
Each dispatch row points at the rate version in force on its date. The notebook recomputes the duty
and flags rows where the stored duty differs from the rate table. State S is synthetic.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2
from calculator import load_rates, row_in_force, duty_for

In [ ]:
fact = pd.read_csv('../data/dispatches_state_s.csv', parse_dates=['dispatch_date'])
dim_sku = pd.read_csv('../data/skus.csv')[['sku_id', 'beer', 'pack', 'bulk_litres_case', 'hl_case', 'abv']]
dim_state = pd.DataFrame({'state': ['S'], 'route_type': ['private wholesale (Route A)']})
dim_rate = load_rates()
dim_date = pd.DataFrame({'date': pd.date_range('2026-04-01', '2026-09-30')})
dim_date['month'] = dim_date.date.dt.to_period('M').astype(str)
print(len(fact), 'dispatch rows;', len(dim_rate), 'rate rows')
dim_rate[['row_id', 'state', 'kind', 'band', 'design', 'share', 'floor_per_bl', 'effective_from', 'effective_to', 'status']]

Join each row to the rate version in force on its dispatch date, by the beer's real ABV, then recompute.

In [ ]:
def in_force(f):
    return row_in_force(dim_rate, f.state, f.abv, f.dispatch_date)
fact['rate_in_force'] = [in_force(f).row_id for f in fact.itertuples()]
fact['duty_due'] = [r2(duty_for(in_force(f), f.dmc_case, f.bulk_litres_case, f.abv) * f.cases) for f in fact.itertuples()]
fact['gap'] = (fact.duty_due - fact.duty_charged).round(2)
flags = fact[(fact.gap.abs() > 0.01) | (fact.rate_version != fact.rate_in_force)]
flags[['pass_no', 'dispatch_date', 'sku_id', 'cases', 'rate_version', 'rate_in_force', 'duty_charged', 'duty_due', 'gap']]

## The chapter's 1,000 case month
September 2026 holds exactly 1,000 cases of SKU 1.

In [ ]:
sep = fact[(fact.sku_id == 'SKU1') & (fact.dispatch_date.dt.month == 9)]
sep_cases, sep_hl = sep.cases.sum(), round(sep.hl.sum(), 3)
sep_duty_due, sep_fee = r2(sep.duty_due.sum()), r2(sep.fee_charged.sum())
print(sep_cases, 'cases =', sep_hl, 'hL; duty due INR', inr(sep_duty_due), '+ fee', inr(sep_fee), '= INR', inr(sep_duty_due + sep_fee))
print('state comparison is a filter: duty due by month')
print(fact.groupby(fact.dispatch_date.dt.to_period('M')).duty_due.sum().map(lambda x: inr(x, 0)))